In [1]:
from huggingface_hub import hf_hub_download, InferenceClient
from langchain_core.messages import SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import serpapi
import json
import requests
import os

d:\predict_volatility\Volatility_predictor\volatile_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
from transformers import pipeline

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [ ]:
Client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["hf_token"]
)

sclient = serpapi.Client(
    api_key=os.getenv("Serpapi")
)

# gclient = ChatGoogleGenerativeAI(
#     model = "gemini-3.7-flash",
#     api_key = os.getenv("GENAI_API_KEY"),
#     temperature = 0
# )
# print(gclient.invoke("Who the president of WHO ??"))

client = ChatOpenAI(
    model = "openrouter/free",
    openai_api_base = "https://openrouter.ai/api/v1",
    openai_api_key = os.getenv("OP_API_KEY")
)

In [4]:
response = client.invoke("Who is the president of WHO")

In [35]:
data = sclient.search({
    "engine": "google_news",
    "q": "Tesla",
    "hl":'en',
    "gl":'in',
    "num": 3
})

results = data.get("news_results", [])

links = []

for article in results[0:3]:
    print("Title", article.get("title"))
    print("Source", article.get("source", {}).get("name"))
    print("Date", article.get("date"))
    links.append(article.get("link"))
    print("Snippet", article.get("snippet"))
    print("\n")

Title Tesla Reportedly Can’t Get the Hands to Work on Its Optimus Robot
Source Futurism
Date 09/28/2026, 05:00 PM, +0000 UTC
Snippet None


Title Tone-Deaf Trillionaire Launches $200,000 Electric Car Into Sour US Market
Source CleanTechnica
Date 09/28/2026, 04:39 AM, +0000 UTC
Snippet None


Title Tesla Robotaxi's Auto-Opening Trunk Feature Wins Praise Online: Video
Source NDTV
Date 09/29/2026, 09:20 AM, +0000 UTC
Snippet None




In [36]:
print(links)

['https://futurism.com/advanced-transport/tesla-cant-get-hands-work-optimus-robot', 'https://cleantechnica.com/2026/09/28/tesla-roadster-electric-sports-car-expensive-aspirational-elections2026/', 'https://www.ndtv.com/auto/tesla-robotaxis-auto-opening-trunk-feature-wins-praise-online-video-12113860']


In [37]:
def summaries(link):
    query = f"""
        Take this {link} and generate summary of the news that is presented in here with focus on the 
        keep it under 50 words and focus on the core message which will hep in understanding if the stock is purchasable or not
    """
    
    response = client.invoke(query)
    return response


def sentimental_analysis(text):
    lst = []
    result = Client.text_classification(
        text,
        model="ProsusAI/finbert"
    )

    for res in result:
        lst.append({res["label"] : res["score"]})

    return lst


In [38]:
for res in results[0:3]:
    Link = res["link"]
    text = res["title"]
    summary = summaries(Link).content
    text += summary
    lst = sentimental_analysis(text)

In [39]:
print(lst)

[{'positive': 0.7106849551200867}, {'neutral': 0.27708208560943604}, {'negative': 0.012232978828251362}]


In [41]:
scores = {}
for item in lst:
    scores.update(item)

pos = scores.get("positive",0)
neg = scores.get("negative",0)
neu = scores.get("neutral",0)

res = pos - neg

if res > 0.15:
    print("positive")
elif res < -0.15:
    print("negative")
else:
    print("nuetral")

positive
